# LELA60331 Week 7 Seminar: Logistic regression, from simulated data to real text

In Week 5 we built logistic regression one step at a time, and got as far as making **one** improvement to the weights (Part 6). This week we:

1. finish the job on the simulated data: **training**, the **learning rate**, **using** the model and **evaluating** it
2. use exactly the same steps to classify **real text**: 10,000 Yelp reviews, with one feature for every word in the vocabulary

| Part | What you do |
|---|---|
| 0 | Run the set-up cell: everything you built in Week 5, Parts 1–6 |
| 1 | Repeat the improvement step many times: **training** |
| 2 | Experiment with the learning rate |
| 3 | Use the trained model |
| 4 | Evaluate it with precision and recall |
| 5 | Matrix-vector multiplication: doing the sums for every text at once |
| 6 | The Yelp reviews |
| 7 | Building a vocabulary |
| 8 | Training and test sets |
| 9 | One-hot encoding |
| 10 | Matrix-vector multiplication with sparse matrices |
| 11 | Training a classifier on the reviews |
| 12 | Evaluating it on the test set |
| 13 | Inspecting the weights, and better tokenisation |

As in Week 5, each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell. Run the check: ✅ means your answer is right, ❌ means not yet. Many activities also tell you what output you should get.

We use **plain Python** (lists, loops, functions and list comprehensions) plus `math`, `random` and `matplotlib`.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Where we got to in Week 5

The cell below sets up everything you built in Week 5, Parts 1–6, so that this sheet works on its own. Read through it to remind yourself of each piece, then run it.

| Name | What it does | Week 5 |
|---|---|---|
| `dot(a, b)` | the dot product of two lists | Week 4 |
| `score(x1, x2, weights, bias)` | `z = bias + x1 * weight1 + x2 * weight2` | Activity 1 |
| `sigmoid(z)` | turns a score into a probability between 0 and 1 | Activity 2 |
| `classify(p)` | 1 if `p` is above 0.5, otherwise 0 | Activity 3 |
| `x`, `y` | the 50 simulated texts: `x[0]` and `x[1]` hold the two features, `y` the labels | Part 3 |
| `loss_one(q, label)`, `average_loss(q, y)` | the cross-entropy loss | Activities 6 and 7 |
| `plot_data()`, `plot_decision_line(weights, bias)` | draw the texts and the decision boundary | Parts 3 and 4 |

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length (from Week 4)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def score(x1, x2, weights, bias):
    return bias + x1 * weights[0] + x2 * weights[1]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def classify(p):
    if p > 0.5:
        return 1
    else:
        return 0

# the simulated data from Week 5, Part 3
random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
num_samples = len(y)

def loss_one(q, label):
    if label == 1:
        return -math.log2(q)
    else:
        return -math.log2(1 - q)

def average_loss(q, y):
    losses = [loss_one(q[k], y[k]) for k in range(len(y))]
    return sum(losses) / len(losses)

def plot_data():
    for k in range(len(y)):
        if y[k] == 1:
            plt.scatter(x[0][k], x[1][k], marker="o", color="tab:orange", s=60)
        else:
            plt.scatter(x[0][k], x[1][k], marker="*", color="tab:blue", s=100)
    plt.xlabel("x1: log count of positive words")
    plt.ylabel("x2: log count of negative words")
    plt.xlim(0, 5)
    plt.ylim(0, 5)

def plot_decision_line(weights, bias):
    plot_data()
    # the line where z = 0, i.e. where the probability is exactly 0.5
    line_x = [0, 5]
    line_y = [-(bias + weights[0] * value) / weights[1] for value in line_x]
    plt.plot(line_x, line_y, "k--")
    plt.show()

print("Ready:", num_samples, "texts,", sum(y), "of them positive")

In Week 5, Part 6 we made **one** improvement to an untrained model:

1. calculate the probability `q` of every text being positive
2. calculate the errors, `q - y`, and from them the gradients `dw1`, `dw2` and `db`
3. move each weight and the bias a little way against its gradient: **new value = old value − learning rate × gradient**

That one step made the loss go down from 2.065 to 1.624. Training simply repeats the step many times.

---
# Part 1 · Training: repeating the step many times

### Activity 1 — The training loop

Everything inside the loop is something you wrote in Week 5, Activities 5–9. Each pass through the loop:

1. calculates the probabilities `q` with the current weights and bias
2. records the loss
3. calculates the errors and the gradients
4. updates the weights and the bias

Fill in the blanks. (One pass through the training data is called an **epoch**.)

In [ ]:
random.seed(10)
weights = [random.random(), random.random()]
bias = 0
lr = 0.5
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):
    # 1. predictions
    q = ???
    # 2. loss
    loss_history.append(???)
    # 3. errors and gradients
    errors = ???
    dw1 = ???
    dw2 = ???
    db = ???
    # 4. update
    weights = ???
    bias = ???

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print("weights:", weights, " bias:", bias)     # about [1.17, -1.32] and 0.51

In [ ]:
# ✅ Check
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("final loss about 0.656", lambda: abs(loss_history[-1] - 0.656) < 0.001)
check("weight for positive words is positive", lambda: weights[0] > 0)
check("weight for negative words is negative", lambda: weights[1] < 0)

The loss falls quickly at first and then levels off. The weights now make sense: positive words push the score up and negative words push it down. Let's look at the new decision boundary.

In [ ]:
plot_decision_line(weights, bias)

### Activity 2 — Accuracy after training

Calculate the accuracy of the trained model, as in Week 5, Activity 5. Before training it was 0.46. How much has it improved? Look at the plot: which texts does it still get wrong, and why can no straight line get them right?

In [ ]:
q = ???
predictions = ???
accuracy = ???
print("accuracy:", accuracy)       # you should get 0.78

In [ ]:
# ✅ Check
check("accuracy", lambda: accuracy == 0.78)

---
# Part 2 · The learning rate

### Activity 3 — Experiment

Go back to Activity 1 and change `lr`, then run the training loop and the cells below it again.

1. Try `lr = 0.01`. What happens to the loss curve? Is 1000 epochs enough?
2. Try `lr = 2`. What happens to the loss curve now? Why?
3. Set `lr` back to 0.5 and run the training loop again before you carry on.

Compare what you see with Activities 2 and 3 of the Week 4 sheet.

*Write your observations here.*

---
# Part 3 · Using the trained model

### Activity 4 — Classifying new texts

Complete `p_positive(n_positive, n_negative)`, which returns the probability that a text with the given **counts** of positive and negative words is positive. Remember that the model's features are the **logs** of the counts, so use `math.log` on each count first.

What is the probability that each of these texts is positive?

a) 2 positive words and 3 negative words
b) 10 positive words and 1 negative word

In [ ]:
def p_positive(n_positive, n_negative):
    x1 = ???
    x2 = ???
    return ???

print(round(p_positive(2, 3), 3))      # you should get 0.466
print(round(p_positive(10, 1), 3))     # you should get 0.961

In [ ]:
# ✅ Check
check("2 positive, 3 negative", lambda: abs(p_positive(2, 3) - 0.466) < 0.005)
check("10 positive, 1 negative", lambda: abs(p_positive(10, 1) - 0.961) < 0.005)

⭐ **Challenge (optional):** what happens if you call `p_positive(0, 3)`? Why? How could you change the features to avoid this problem?

---
# Part 4 · Evaluation: precision and recall

Accuracy can be misleading. If 90% of texts were negative, a model that always said "negative" would have an accuracy of 0.9, but it would never find a single positive text. So we also measure **precision** and **recall** for the positive class:

* a **true positive** is a positive text that the model predicted as positive
* a **false positive** is a negative text that the model wrongly predicted as positive
* a **false negative** is a positive text that the model wrongly predicted as negative

**Precision** = true positives / (true positives + false positives): of the texts the model *called* positive, how many really were?

**Recall** = true positives / (true positives + false negatives): of the texts that really *were* positive, how many did the model find?

### Activity 5 — Precision and recall

Complete the counting loop and the two calculations, using `predictions` from Activity 2. The condition for true positives is done for you.

In [ ]:
tp = 0
fp = 0
fn = 0
for k in range(num_samples):
    if predictions[k] == 1 and y[k] == 1:
        tp += 1
    if ???:          # predicted positive, but really negative
        fp += 1
    if ???:          # predicted negative, but really positive
        fn += 1

precision = ???
recall = ???
print(tp, fp, fn)                  # you should get 18 6 5
print("precision:", round(precision, 3), " recall:", round(recall, 3))

In [ ]:
# ✅ Check
check("the counts add up", lambda: tp + fn == sum(y))
check("precision", lambda: precision == tp / (tp + fp))
check("recall", lambda: recall == tp / (tp + fn))

Which is higher, precision or recall? What does that tell you about the mistakes this model makes?

That completes logistic regression on the simulated data. For the rest of the sheet we use exactly the same steps on real text.

---
# Part 5 · Matrix-vector multiplication

So far each text has had just **two** features, and we have written a separate piece of code for each one: `x1 * weight1 + x2 * weight2`, and separate gradients `dw1` and `dw2`. Real text needs far more features: in Part 9 each review will have **5000**, one for every word in the vocabulary. We can't write 5000 lines of code, so we need a way of doing the sums for **all** the features, and **all** the texts, at once.

## Matrices

A **matrix** is a table of numbers with rows and columns. In plain Python we store a matrix as a **list of lists**, with one inner list for each row (Week 4 Python workbook). `matrix[i]` is row `i` and `matrix[i][j]` is the element in row `i`, column `j`. A vector is just a list.

For our data, the natural matrix has **one row per text** and **one column per feature**: a 50 × 2 matrix. Text `k`'s row is `[x[0][k], x[1][k]]`.

### Activity 6 — The data as a matrix

Use a list comprehension to make `X`, a list with one row `[x1, x2]` for each text.

In [ ]:
X = ???

print(len(X), len(X[0]))      # you should get 50 2: 50 rows, 2 columns
print(X[0])                   # text 0's two features
print(X[0][1])                # row 0, column 1: text 0's second feature

In [ ]:
# ✅ Check
check("50 rows", lambda: len(X) == 50)
check("2 columns", lambda: all(len(row) == 2 for row in X))
check("row 0", lambda: X[0] == [x[0][0], x[1][0]])
check("row 49", lambda: X[49] == [x[0][49], x[1][49]])

## Multiplying a matrix by a vector

**Matrix-vector multiplication** takes the dot product of **each row** of the matrix with the vector, giving one number per row. For example:

$$\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix} \begin{bmatrix} 1 \\ 0 \\ 2 \end{bmatrix} = \begin{bmatrix} 1 \times 1 + 2 \times 0 + 3 \times 2 \\ 4 \times 1 + 5 \times 0 + 6 \times 2 \end{bmatrix} = \begin{bmatrix} 7 \\ 16 \end{bmatrix}$$

With our data matrix `X` and the vector `weights`, each row's dot product is `x1 * weight1 + x2 * weight2`: exactly the score of that text, without the bias. So one matrix-vector multiplication gives us (almost) the score of every text at once, however many features there are.

### Activity 7 — A matrix-vector multiplication function

1. Complete `mat_vec(matrix, vector)`, which returns a list with the dot product of each row of `matrix` with `vector`.
2. Use it to calculate `z_all`: the score of every text, using the trained `weights` and `bias` from Part 1. *Hint:* add the bias to each number that `mat_vec` gives you.

In [ ]:
def mat_vec(matrix, vector):
    result = []
    for row in matrix:
        result.append(???)
    return result

print(mat_vec([[1, 2, 3], [4, 5, 6]], [1, 0, 2]))     # you should get [7, 16]

z_all = ???
print(len(z_all))                                     # you should get 50

In [ ]:
# ✅ Check
check("mat_vec on the example", lambda: mat_vec([[1, 2, 3], [4, 5, 6]], [1, 0, 2]) == [7, 16])
check("z_all has one score per text", lambda: len(z_all) == 50)
check("z_all gives the same scores as score()",
      lambda: all(abs(z_all[k] - score(x[0][k], x[1][k], weights, bias)) < 1e-9 for k in range(50)))

### Shapes matter

Each row of `X` has 2 elements, so `X` can only be multiplied by a vector with 2 elements. Here is a version of `dot` that checks this and gives an error if the lengths don't match. Run the cell: the second multiplication fails, because `[1, 2, 3]` has 3 elements.

In [ ]:
def dot(a, b):
    # the dot product of two lists of the same length
    if len(a) != len(b):
        raise ValueError("the two lists must have the same length")
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

print(mat_vec(X, [1, 2])[:3])          # fine: 2 columns, 2 weights
try:
    mat_vec(X, [1, 2, 3])              # 2 columns, but 3 weights
except ValueError as err:
    print("ValueError:", err)

## The gradients are a matrix-vector multiplication too

In Week 5 we calculated the gradients like this:

* `dw1 = dot(x[0], errors) / num_samples`
* `dw2 = dot(x[1], errors) / num_samples`

Look at how our data is stored. `x` has one row per **feature**: `x[0]` holds feature 1 for every text and `x[1]` holds feature 2. `X` has one row per **text**. So `x` is `X` with its rows turned into columns. This is called the **transpose** of `X`. Each gradient is the dot product of one row of `x` with the errors, so **all** the gradients together are the matrix-vector multiplication of `x` with `errors`.

### Activity 8 — All the gradients at once

Using the trained model's probabilities `q` from Activity 2:

1. `errors`: `q - y` for every text (as in Week 5)
2. `gradients`: a list with one gradient per weight, `[dw1, dw2]`, calculated with **one** call to `mat_vec`, then dividing each number by `num_samples`

In [ ]:
errors = ???
gradients = ???
print(gradients)          # both are close to 0: the trained model is near the bottom of the loss

In [ ]:
# ✅ Check
check("two gradients", lambda: len(gradients) == 2)
check("gradient for weight 1", lambda: abs(gradients[0] - dot(x[0], errors) / num_samples) < 1e-9)
check("gradient for weight 2", lambda: abs(gradients[1] - dot(x[1], errors) / num_samples) < 1e-9)

So training needs just two matrix-vector multiplications, whatever the number of features:

* **matrix (one row per text) × weights** → a score for every text
* **transpose (one row per feature) × errors** → a gradient for every weight

We will use exactly these two steps on the Yelp reviews.

---
# Part 6 · The Yelp reviews

Our data set is 10,000 reviews from Yelp, each labelled **negative** (1 or 2 stars) or **positive** (3 or 4 stars). We will train a classifier on some of them and test how well it does on the rest.

First we download the data set. A line beginning with `!` is a command for the computer, not Python: `wget` downloads a file from the web.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

The file is **tab-separated**: each line holds a review, then a tab, then the label. The first line is a header. `head -n2` shows the first two lines:

In [ ]:
!head -n2 yelp_reviews.txt

The text has already been lower-cased and its punctuation separated from the words by spaces, so that it is easy to split into words. (The odd `n n` is what is left of the line breaks.)

This cell reads the file into two lists: `reviews` holds the texts and `labels` the labels, so review `i` has the label `labels[i]`. Just run it.

In [ ]:
reviews = []
labels = []

with open("yelp_reviews.txt") as fp:
    lines = fp.readlines()
for line in lines[1:]:                         # skip the header line
    fields = line.rstrip().split("\t")         # [review, label]
    reviews.append(fields[0])
    labels.append(fields[1])

### Activity 9 — Getting to know the reviews

1. `n_reviews`: the number of reviews.
2. `n_positive_reviews`: the number of positive reviews. *Hint:* the labels are strings this time, so use the list method `count`.
3. `first_review_length`: the number of words in review 0. *Hint:* split it at the spaces.

Then read review 0. Does its label make sense?

In [ ]:
n_reviews = ???
n_positive_reviews = ???
first_review_length = ???
print(n_reviews, n_positive_reviews, first_review_length)      # you should get 10000 5000 118
print(labels[0], ":", reviews[0])

In [ ]:
# ✅ Check
check("n_reviews", lambda: n_reviews == 10000)
check("n_positive_reviews", lambda: n_positive_reviews == 5000)
check("first_review_length", lambda: first_review_length == 118)

Are the classes balanced? What accuracy would a model get if it always said "positive"?

---
# Part 7 · Building a vocabulary

Our features will be the words. Each review gets one feature for every word in a **vocabulary**: 1 if the word occurs in the review and 0 if it doesn't. We use the 5000 most frequent words as our vocabulary (not an ideal solution, but a convenient one).

First we **tokenise** each review, splitting it into a list of words. This is a very naive tokenisation that just splits the text at spaces; we will come back to it in Part 13. Run the cell.

In [ ]:
tokenized_sents = []
for review in reviews:
    tokenized_sents.append(review.split())

print(len(tokenized_sents))         # one list of words per review
print(tokenized_sents[0][:12])      # the first 12 words of review 0

### Activity 10 — Counting the words

Complete the loop so that `counts` becomes a **counting dictionary** (Week 3 seminar): each word is a key and its value is the number of times the word occurs across all the reviews. *Hint:* `counts.get(word, 0)` gives the count so far, or 0 if the word hasn't been seen yet.

In [ ]:
counts = {}
for sent in tokenized_sents:
    for word in sent:
        counts[word] = ???

print(len(counts))                              # the number of different words: you should get 28495
print(counts["good"], counts["bad"])            # you should get 5012 1285

In [ ]:
# ✅ Check
check("number of different words", lambda: len(counts) == 28495)
check("count of 'good'", lambda: counts["good"] == 5012)
check("count of 'bad'", lambda: counts["bad"] == 1285)

### Activity 11 — The 5000 most frequent words

To find the most frequent words, we make a list of `(count, word)` pairs and sort it from largest to smallest, as in the Week 3 seminar. Python sorts pairs by their first element, the count.

Complete `type_list`: a list of just the **words** from the first 5000 pairs. *Hint:* a list comprehension over `pairs[:5000]` that unpacks each pair into `count, word`.

In [ ]:
pairs = [(count, word) for word, count in counts.items()]
pairs = sorted(pairs, reverse=True)
print(pairs[:5])

type_list = ???
print(len(type_list))          # you should get 5000
print(type_list[:10])          # the 10 most frequent words

In [ ]:
# ✅ Check
check("5000 words", lambda: len(type_list) == 5000)
check("the most frequent word first", lambda: type_list[0] == pairs[0][1])
check("only words, not pairs", lambda: all(type(word) == str for word in type_list))

What kinds of words are most frequent? Do you think they will be useful for telling positive from negative reviews?

Each word in the vocabulary gets its own feature, and so its own **column** in our data matrix: `type_list[j]` is the word for column `j`. We also need to go the other way, from a word to its column, so we make a dictionary `word_to_index`. Run the cell.

In [ ]:
word_to_index = {}
for j in range(len(type_list)):
    word_to_index[type_list[j]] = j

print(word_to_index["good"], type_list[word_to_index["good"]])

---
# Part 8 · Training and test sets

We want to know how well our classifier does on reviews it has **never seen**. So we split the data: we train on 80% of the reviews (8000) and test on the other 20% (2000). In a real project we would also keep a third set, a **development** set, for trying out different settings; here we just use training and test.

`random.sample(sequence, k)` picks `k` different elements at random from `sequence`. We use it to pick the indices of 8000 training reviews. The test reviews are all the indices that were not picked. Run the cell.

In [ ]:
random.seed(1)        # so that everyone gets the same split
train_ints = random.sample(range(len(reviews)), int(len(reviews) * 0.8))

# The test items are all the indices that were not picked for training.
# Checking whether something is in a set is very fast.
train_set = set(train_ints)
test_ints = []
for i in range(len(reviews)):
    if i not in train_set:
        test_ints.append(i)

print(len(train_ints), len(test_ints))
print(train_ints[:5])

### Activity 12 — Labels as numbers

Our model works with labels 1 (positive) and 0 (negative), as in Week 5. Use list comprehensions to make:

1. `y_train`: the label of each training review, as 1 or 0, in the order of `train_ints`
2. `y_test`: the same for the test reviews

*Hint:* `int(labels[i] == "positive")` is 1 if review `i` is positive and 0 if not, because `int(True)` is 1 and `int(False)` is 0.

In [ ]:
y_train = ???
y_test = ???
print(len(y_train), sum(y_train))       # you should get 8000 4026
print(len(y_test), sum(y_test))         # you should get 2000 974

In [ ]:
# ✅ Check
check("y_train", lambda: len(y_train) == 8000 and sum(y_train) == 4026)
check("y_test", lambda: len(y_test) == 2000 and sum(y_test) == 974)
check("no review is in both sets", lambda: len(set(train_ints) & set(test_ints)) == 0)

---
# Part 9 · One-hot encoding

Now we turn each review into numbers. We want a matrix **M_train** with one row per training review (8000) and one column per vocabulary word (5000), and a matrix **M_test** in the same way for the 2000 test reviews. Element $x_{i,j}$ is 1 if word $j$ occurs in review $i$, and 0 otherwise. This is called **one-hot encoding**.

### Sparse rows

As a list of lists, M_train would hold 8000 × 5000 = 40 million numbers. That is very slow to work with in plain Python, and almost all of them are 0: a review contains only a small fraction of the 5000 vocabulary words. So instead we use a **sparse** representation: for each review we store only the **columns that hold a 1**, as a list of column numbers.

### Activity 13 — A tiny example

Here is a vocabulary of just 4 words and a review. Complete the loop so that it builds both:

* `full_row`: the full row of 0s and 1s
* `sparse_row`: the list of the columns that hold a 1

In [ ]:
tiny_vocab = ["good", "bad", "food", "service"]      # columns 0, 1, 2 and 3
review = "good food good".split()

full_row = [0, 0, 0, 0]
sparse_row = []
for j in range(len(tiny_vocab)):
    if ???:
        full_row[j] = 1
        sparse_row.append(???)
print(full_row)      # you should get [1, 0, 1, 0]
print(sparse_row)    # you should get [0, 2]

In [ ]:
# ✅ Check
check("full_row", lambda: full_row == [1, 0, 1, 0])
check("sparse_row", lambda: sparse_row == [0, 2])

"good" occurs twice in the review, but its column holds a 1, not a 2: we only record whether a word is **present**.

Why does the sparse row give the same answers? The dot product of a row of 0s and 1s with a vector adds up the vector's elements wherever the row has a 1, and ignores the rest (they are multiplied by 0). The sparse row lists exactly those positions. Run the cell to see that the two give the same result.

In [ ]:
tiny_weights = [0.5, -2.0, 0.25, 1.0]

total = 0
for j in sparse_row:
    total += tiny_weights[j]
print(dot(full_row, tiny_weights), total)       # both 0.75

### Activity 14 — One-hot encoding the reviews

We need to build two matrices in the same way, so we write a **function** for it. `one_hot(token_lists)` takes a list of tokenised reviews and returns their sparse matrix: a list with one row per review, where each row is the sorted list of the columns that hold a 1.

A word can occur several times in a review, but its column should only be stored once, so each row is first collected in a **set**. For each word in the review, fill in:

1. the condition: only words that are in our vocabulary have a column (`word_to_index` contains exactly those words)
2. the column for that word

Encoding takes a few seconds.

In [ ]:
def one_hot(token_lists):
    M = []
    for tokens in token_lists:
        row = set()
        for word in tokens:
            if ???:
                row.add(???)
        M.append(sorted(row))       # the columns that hold a 1, in order
    return M

# the tokenised reviews for the training items and for the test items
train_tokens = [tokenized_sents[i] for i in train_ints]
test_tokens = [tokenized_sents[i] for i in test_ints]

M_train = one_hot(train_tokens)
M_test = one_hot(test_tokens)
print(len(M_train), len(M_test))      # the number of rows (reviews) in each matrix: 8000 2000
print(M_train[0][:10])                # the first 10 columns that hold a 1 in the first training review
print([type_list[j] for j in M_train[0][:10]])     # ... and their words

In [ ]:
# ✅ Check
check("8000 training rows and 2000 test rows", lambda: len(M_train) == 8000 and len(M_test) == 2000)
check("the first training review",
      lambda: M_train[0] == sorted(set(word_to_index[w] for w in train_tokens[0] if w in word_to_index)))
check("each column stored once, in order", lambda: all(row == sorted(set(row)) for row in M_train[:100]))
check("every column is between 0 and 4999", lambda: all(0 <= j < 5000 for row in M_test for j in row))

### Activity 15 — How sparse?

Calculate `average_ones`: the average number of 1s in a row of `M_train` (that is, the average number of different vocabulary words in a training review). What proportion of the 5000 columns is that?

In [ ]:
average_ones = ???
print(round(average_ones, 1))                   # you should get 78.5
print(round(average_ones / 5000, 3))            # the proportion of columns that hold a 1

In [ ]:
# ✅ Check
check("average_ones", lambda: abs(average_ones - 78.5) < 0.01)

---
# Part 10 · Matrix-vector multiplication with sparse matrices

At the end of Part 5 we saw that training needs two matrix-vector multiplications:

* **matrix × weights** → a score for every review
* **transpose × errors** → a gradient for every weight

We now write both for our sparse matrices.

### Activity 16 — Scores: sparse matrix × vector

`sparse_mat_vec(M, vector)` does the same job as `mat_vec` in Part 5, but for a sparse matrix. For each row it adds up `vector`'s elements at the columns that hold a 1, just like the tiny example in Part 9. Complete the line inside the inner loop.

In [ ]:
def sparse_mat_vec(M, vector):
    result = []
    for row in M:
        total = 0
        for j in row:
            total += ???
        result.append(total)
    return result

print(sparse_mat_vec([sparse_row], tiny_weights))                    # you should get [0.75]
print(sparse_mat_vec([[0, 2], [1, 2], [3]], tiny_weights))           # you should get [0.75, -1.75, 1.0]

In [ ]:
# ✅ Check
check("one row", lambda: sparse_mat_vec([sparse_row], tiny_weights) == [0.75])
check("three rows", lambda: sparse_mat_vec([[0, 2], [1, 2], [3]], tiny_weights) == [0.75, -1.75, 1.0])
check("same as mat_vec with full rows",
      lambda: sparse_mat_vec([[0, 2], [1, 2]], tiny_weights) == mat_vec([[1, 0, 1, 0], [0, 1, 1, 0]], tiny_weights))

### Activity 17 — Gradients: transpose × vector

For the gradients, Week 5 used `dot(x[0], errors)`: for each **feature**, multiply its value in each text by that text's error, and add up. With one-hot features the value is 1 or 0, so the gradient for word `j` is simply **the sum of the errors of the reviews that contain word `j`**.

`sparse_transpose_mat_vec(M, vector, num_columns)` calculates this for every column at once. It starts with a 0 for every column, then goes through the rows: for each column `j` that holds a 1 in row `i`, it adds `vector[i]` (row `i`'s error) to column `j`'s total. Complete the line inside the inner loop.

*Example:* with the two reviews `[[0, 2], [1, 2]]` and errors `[10, 1]`, column 0 gets 10 (only review 0 has it), column 1 gets 1, column 2 gets 10 + 1 = 11 (both reviews have it), and column 3 gets 0.

In [ ]:
def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):
        for j in M[i]:
            result[j] += ???
    return result

print(sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4))     # you should get [10, 1, 11, 0]

In [ ]:
# ✅ Check
check("the example", lambda: sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4) == [10, 1, 11, 0])
check("same as mat_vec with the transposed full rows",
      lambda: sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4) == mat_vec([[1, 0], [0, 1], [1, 1], [0, 0]], [10, 1]))

---
# Part 11 · Training a classifier on the reviews

We now have everything we need. The model works exactly like the two-feature model in Part 1, but with 5000 weights, one for each word.

### Two small changes

**1. A safer sigmoid.** With 5000 features, `z` can be a very large positive or negative number. `math.exp(-z)` is then too big for Python to store when `z` is a large negative number. This version of `sigmoid` uses an equivalent formula for negative `z` that avoids the problem. It gives exactly the same values.

**2. A safer loss.** When `z` is large, the sigmoid can round to exactly 1 (or 0). If the right answer was given a probability of 0, `math.log2(0)` gives an error. So we add a tiny number, `eps`, inside the log.

Run the cell.

In [ ]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

eps = 0.00001          # a tiny number, so that we never take the log of 0

def loss_one(q, label):
    if label == 1:
        return -math.log2(q + eps)
    else:
        return -math.log2(1 - q + eps)

print(sigmoid(-1000), sigmoid(0), sigmoid(1000))      # 0.0 0.5 1.0, and no error
print(loss_one(1.0, 0))                               # a big loss, but no error

### Activity 18 — Predictions from an untrained model

As in Week 5, the model starts with random weights (and here a random bias too). Calculate, for every training review:

1. `z`: the score. *Hint:* `sparse_mat_vec` with `M_train` and `weights`, then add the bias to each number, as in Activity 7.
2. `q`: the probability, the sigmoid of each score
3. `predictions`: the class (1 or 0)
4. `accuracy`: the proportion of predictions that equal the labels `y_train`

Then print the loss with `average_loss`.

In [ ]:
random.seed(2)
num_features = len(type_list)
weights = [random.random() for j in range(num_features)]
bias = random.random()
num_samples = len(y_train)

z = ???
q = ???
predictions = ???
accuracy = ???
print(round(z[0], 3), round(q[0], 3), predictions[0])     # you should get 25.646 1.0 1
print("accuracy:", accuracy)                              # you should get 0.50325
print("loss:", round(average_loss(q, y_train), 3))        # you should get 8.071

In [ ]:
# ✅ Check
check("z", lambda: len(z) == 8000 and abs(z[0] - 25.646) < 0.001)
check("q", lambda: len(q) == 8000 and abs(q[0] - 1.0) < 0.001)
check("predictions", lambda: predictions[:5] == [classify(p) for p in q[:5]])
check("accuracy", lambda: accuracy == 0.50325)

Every weight starts positive, so every word pushes the score up. What does the model predict for every review? So why is the accuracy the number it is? (Look back at your answer to Activity 12.)

### Activity 19 — One step of gradient descent

Now make one improvement, as in Week 5, Activities 8 and 9:

1. `errors`: `q - y_train` for every review
2. `dw`: the gradient for every weight. *Hint:* `sparse_transpose_mat_vec` with `M_train`, `errors` and `num_features`, then divide each number by `num_samples`.
3. `db`: the sum of the errors divided by the number of reviews
4. `new_weights` and `new_bias`: **new value = old value − learning rate × gradient**. *Hint:* for `new_weights`, a list comprehension over the positions `j` of the weights.
5. `new_q`: the probabilities with the new weights and bias

Has the loss gone down?

In [ ]:
lr = 0.4

errors = ???
dw = ???
db = ???

new_weights = ???
new_bias = ???

new_q = ???
print(round(dw[0], 3), round(db, 3))                                # you should get 0.488 0.496
print("loss before:", round(average_loss(q, y_train), 3))           # 8.071
print("loss after: ", round(average_loss(new_q, y_train), 3))       # you should get 8.007

In [ ]:
# ✅ Check
check("errors", lambda: len(errors) == 8000 and abs(errors[0] - (q[0] - y_train[0])) < 1e-9)
check("dw", lambda: len(dw) == 5000 and abs(dw[0] - 0.4884) < 0.001)
check("db", lambda: abs(db - 0.4957) < 0.001)
check("new_bias", lambda: abs(new_bias - (bias - 0.4 * db)) < 1e-9)
check("the loss went down", lambda: average_loss(new_q, y_train) < average_loss(q, y_train))

Word 0 is `type_list[0]`, the most frequent word. Why is its gradient so close to `db`? *Hint:* which reviews contain it?

### Activity 20 — The training loop

Now put the steps from Activities 18 and 19 into a loop, as in Activity 1. Each pass through the loop:

1. calculates the probabilities `q` with the current weights and bias
2. records the loss
3. calculates the errors and the gradients
4. updates the weights and the bias

Training for 500 epochs takes about a minute in plain Python. While you wait, think about why each epoch takes so much longer than in Part 1.

In [ ]:
random.seed(2)
weights = [random.random() for j in range(num_features)]
bias = random.random()
lr = 0.4
n_epochs = 500
loss_history = []

for epoch in range(n_epochs):
    # 1. predictions
    q = ???
    # 2. loss
    loss_history.append(???)
    # 3. errors and gradients
    errors = ???
    dw = ???
    db = ???
    # 4. update
    weights = ???
    bias = ???
    if epoch % 50 == 0:
        print("epoch", epoch, " loss", round(loss_history[-1], 3))

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print("final loss:", round(loss_history[-1], 3))      # you should get 0.598

In [ ]:
# ✅ Check
check("500 epochs", lambda: len(loss_history) == 500)
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("final loss about 0.598", lambda: abs(loss_history[-1] - 0.598) < 0.005)

The first epoch's loss is very high: the untrained model gives almost every negative review a probability of nearly 1 of being positive. Compare the shape of the curve with Part 1.

---
# Part 12 · Evaluating on the test set

Now we see how well the model does on the 2000 test reviews, which it never saw during training.

### Activity 21 — Predictions for the test reviews

Calculate, for every test review, as in Activity 18 but with `M_test` and `y_test`:

1. `q_test`: the probability that it is positive
2. `y_test_pred`: the predicted class (1 or 0)
3. `accuracy_test`: the proportion of predictions that equal the labels

In [ ]:
q_test = ???
y_test_pred = ???
accuracy_test = ???
print("test accuracy:", accuracy_test)        # you should get 0.82

In [ ]:
# ✅ Check
check("q_test", lambda: len(q_test) == 2000 and all(0 <= p <= 1 for p in q_test))
check("y_test_pred", lambda: y_test_pred[:10] == [classify(p) for p in q_test[:10]])
check("accuracy_test", lambda: abs(accuracy_test - 0.82) < 0.001)

### Activity 22 — Precision and recall as functions

We will want precision and recall for the test set **and** for the training set, so this time we write them as **functions**, using the same counting as in Activity 5.

1. Complete `confusion_counts(predicted, actual)`, which counts the true positives, false positives and false negatives. The condition for true positives is given.
2. Complete `precision(tp, fp)` and `recall(tp, fn)`.

A function can `return` several values, separated by commas. They come back as a tuple, which we can unpack into separate names: `tp, fp, fn = confusion_counts(...)` (Week 4 Python workbook).

In [ ]:
def confusion_counts(predicted, actual):
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(predicted)):
        if predicted[k] == 1 and actual[k] == 1:
            tp += 1
        if ???:          # predicted positive, but really negative
            fp += 1
        if ???:          # predicted negative, but really positive
            fn += 1
    return tp, fp, fn

def precision(tp, fp):
    return ???

def recall(tp, fn):
    return ???

tp, fp, fn = confusion_counts(y_test_pred, y_test)
print(tp, fp, fn)                                        # you should get 807 193 167
print("test precision:", round(precision(tp, fp), 3))    # you should get 0.807
print("test recall:", round(recall(tp, fn), 3))          # you should get 0.829

In [ ]:
# ✅ Check
check("confusion_counts on a small example", lambda: confusion_counts([1, 1, 0, 0, 1], [1, 0, 1, 0, 1]) == (2, 1, 1))
check("precision", lambda: precision(2, 1) == 2 / 3)
check("recall", lambda: recall(2, 1) == 2 / 3)
check("the test counts add up", lambda: tp + fn == sum(y_test))

### Activity 23 — Performance on the training data

Now calculate precision and recall on the **training** data. You need the predicted classes for the training reviews first (as in Activity 21, but with `M_train`); then use your functions from Activity 22.

How do the numbers differ from those for the test set? Why do you think this is?

In [ ]:
q_train = ???
y_train_pred = ???
tp, fp, fn = ???
print("training precision:", round(precision(tp, fp), 3))     # you should get 0.857
print("training recall:", round(recall(tp, fn), 3))           # you should get 0.862

In [ ]:
# ✅ Check
check("training predictions", lambda: len(y_train_pred) == 8000)
check("training counts", lambda: (tp, fp, fn) == confusion_counts(y_train_pred, y_train))
check("better on the training data than on the test data",
      lambda: precision(tp, fp) > precision(*confusion_counts(y_test_pred, y_test)[:2]))

*Write your answer here.*

A model that does much better on its training data than on new data has **overfitted**: it has learned things that are true of the particular reviews it was trained on, but that don't hold in general.

---
# Part 13 · Inspecting the weights

With one-hot features, the weights are easy to interpret. Each weight belongs to one word: `weights[j]` is the weight for the word `type_list[j]`. Words with **high** weights are those whose presence makes the model most likely to say *positive*; words with **low** (very negative) weights are those whose presence makes it most likely to say *negative*.

To put the words in order of their weights, we make a list of `(weight, word)` pairs and sort it, just as we sorted `(count, word)` pairs in Part 7. This time we **don't** reverse it, so it goes from the lowest weight to the highest. Run the cell.

In [ ]:
weight_word = [(weights[j], type_list[j]) for j in range(len(type_list))]
weight_word = sorted(weight_word)
print(weight_word[:3])         # the three lowest weights
print(weight_word[-3:])        # the three highest weights

### Activity 24 — The most negative and most positive words

1. `lowest`: a list of the **words** with the 20 lowest weights
2. `highest`: a list of the **words** with the 20 highest weights, starting with the highest. *Hint:* the slice `[::-1]` reverses a list.

In [ ]:
lowest = ???
highest = ???
print(lowest)
print(highest)

In [ ]:
# ✅ Check
check("20 lowest", lambda: len(lowest) == 20 and lowest[0] == weight_word[0][1])
check("20 highest, highest first", lambda: len(highest) == 20 and highest[0] == weight_word[-1][1])
check("words, not pairs", lambda: type(lowest[0]) == str and type(highest[0]) == str)

Do the lists make sense? Which words would you have expected to see?

### Activity 25 — Spotting overfitting

Look further down the lists: print the 100 lowest and 100 highest words. Do you spot any words that suggest overfitting: words that tell the model about these particular reviews, rather than about sentiment in general? *Hint:* look for numbers, names, and tokens that aren't really words.

In [ ]:
# your code here

*Write your observations here.*

### Activity 26 — Better tokenisation

Go back to the cell at the start of Part 7 where we tokenise the reviews (where `tokenized_sents` is built). Can you change the way the text is tokenised to get rid of the problematic tokens you noted in Activity 25? The string methods from the Week 2 Python workbook, such as `lower()`, `replace()`, `strip()` and `isalpha()`, will help.

Then run all the cells from there down to here again (the check cells for the counts in Part 7 will now show ❌, because the words have changed). How do your changes affect the test precision and recall, and the lists of words?

*Write your observations here.*

---
## ✏️ Summary

You have finished building logistic regression, and used it on real text:

1. **training**: repeat the improvement step many times, with a sensible learning rate
2. **evaluation**: accuracy, precision and recall, on data the model has never seen
3. **matrix-vector multiplication**: a score for every text is the data matrix × the weights, and a gradient for every weight is the transpose × the errors, however many features there are
4. **text as features**: tokenise, choose a vocabulary, and one-hot encode each text, storing only the columns that hold a 1 (a **sparse** matrix)
5. **overfitting**: a model can do much better on its training data than on new data; inspecting the weights helps show why